In [0]:
import pandas as pd 
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col
import re


spark = SparkSession.builder.getOrCreate()


base ="/Volumes/investment_intelligence_platform/ingestion/api_raw/"
output ="investment_intelligence_platform.bronze"


# yfinance has its own base path
yfinance_base = base + "yfinance/"
companies_house_base = base + "companies_house/"

def clean_column_names(df):
    new_columns = [
        re.sub(r'[^a-zA-Z0-9_]', '_', col)  # replace anything bad with _
        for col in df.columns
    ]
    return df.toDF(*new_columns)
def read_json(path):
    df = spark.read \
        .option("multiline", "true") \
        .option("recursiveFileLookup", "true") \
        .json(path)
    df = df.withColumn("last_updated_ts", current_timestamp()) \
        .withColumn("meta_data", col("_metadata.file_path"))
    df = clean_column_names(df) 
    return df

# api_raw folders
company_df        = read_json(companies_house_base + "company/")
filing_history_df = read_json(companies_house_base + "filing-history/")
officers_df       = read_json(companies_house_base + "officers/")

# yfinance folders 
balance_sheet_df     = read_json(yfinance_base + "balance_sheet/")
cashflow_df          = read_json(yfinance_base + "cashflow/")

income_statement_df  = read_json(yfinance_base + "income_statement/")
stats_df             = read_json(yfinance_base + "stats/")


#  use this for bronze schema
def save_tables(df, table_name):
    full_name = f"investment_intelligence_platform.bronze.{table_name}"
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_name)
    print(f" saved → {full_name}")


# call for all tables
save_tables(company_df,          "company")
save_tables(filing_history_df,   "filing_history")
save_tables(officers_df,         "officers")
save_tables(balance_sheet_df,    "balance_sheet")
save_tables(cashflow_df,         "cashflow")

save_tables(income_statement_df, "income_statement")
save_tables(stats_df,            "stats")